# PatchSorter — Database Initialization

End-to-end setup for a new PatchSorter database:

1. Create schema and distribute tables (Citus reference tables)
2. Seed application-level settings
3. Create a placeholder project (project-level settings seeded automatically)
4. Create per-project distributed tables and install confusion-matrix triggers
5. Add 10 label classes to the project
6. Register one whole-slide image
7. Extract patches from a GeoJSON file and load them into the database

> **Prerequisites:**
> - Docker stack running: `docker-compose -f deployment/docker-compose.yaml up -d`
> - GeoJSON features contain a `uid` field (run `add_uuids_to_geojson.ipynb` first if needed)
> - `IMAGE_FILEPATH` points to a `large_image`-readable whole-slide image

In [1]:
# ---------------------------------------------------------------------------
# Configuration — edit this cell before running
# ---------------------------------------------------------------------------

IMAGE_FILEPATH      = "/opt/PatchSorter/mounts/fan_annotations/13_266069_040_003_L02_PAS.ndpi"       # large_image-readable WSI
GEOJSON_FILEPATH    = "/opt/PatchSorter/mounts/fan_annotations/nuclei_split.geojson"  # must have 'uid' on each feature

PROJECT_NAME        = "Placeholder Project"
PROJECT_DESCRIPTION = "Initial placeholder project for development"

DEEPZOOM_TILESIZE = 256     # tile size for DeepZoom serving

# Patch extraction parameters
DOWNSAMPLE_FACTOR = 2.0  # 1.0 = base mag, 2.0 = half base mag, etc.
PATCH_SIZE        = 64  # output patch edge length in pixels at extraction mag

# Ten cell subtype label classes: (name, CSS hex colour)
LABEL_CLASSES = [
    ("Epithelial",   "#E74C3C"),
    ("Lymphocyte",   "#3498DB"),
    ("Plasma Cell",  "#9B59B6"),
    ("Macrophage",   "#E67E22"),
    ("Neutrophil",   "#F1C40F"),
    ("Eosinophil",   "#1ABC9C"),
    ("Fibroblast",   "#2ECC71"),
    ("Endothelial",  "#E91E63"),
    ("Mast Cell",    "#884400"),
    ("Tumor Cell",   "#C0392B"),
]

In [2]:
from patchsorter.db.head_client import (
    get_client,
    ImageStore,
    LabelClassStore,
    ProjectStore,
)
from patchsorter.db.head_client.database_manager import DatabaseManager
from patchsorter.utils.patch_extraction import _makepatch_geojson

In [3]:
# Connect to the head node and initialise the schema.
# setup_schema() creates all base tables, distributes them as Citus reference
# tables, seeds the reserved "unassigned" label class, and seeds application-
# level settings from settings_defaults.toml.
client = get_client()
db_mgr = DatabaseManager(client)


In [4]:
db_mgr.drop_all_tables()  # drop existing tables for a clean slate


Dropped public.project1_patch
Dropped public.project1_pred_patch_latest
Dropped public.project1_pred_patch_last
Dropped public.project1_confusion_matrix_l8
Dropped public.project1_confusion_matrix_l9
Dropped public.project1_confusion_matrix_l10
Dropped public.project1_confusion_matrix_l11
Dropped public.project1_confusion_matrix_l12


In [5]:
db_mgr.setup_schema()
print("Schema ready.")

Schema ready.


In [6]:
# Create the project.  ProjectStore.create() also seeds project-scoped
# settings (world_size, agg_hierarchy_depth) via SettingsStore.seed_project_settings.
with client.get_session() as session:
    project_store = ProjectStore(session)
    proj = project_store.create(PROJECT_NAME, PROJECT_DESCRIPTION)

project_id = proj["project_id"]
print(f"Project created  project_id={project_id}  name={proj['project_name']}")

# Create per-project distributed tables (patch, pred_patch_latest/last,
# confusion_matrix_l8..l12) and install per-shard triggers.
db_mgr.setup_project(project_id)
print(f"Per-project tables and triggers ready for project {project_id}.")

Project created  project_id=1  name=Placeholder Project
Ensured existence of table project1_patch for project 1.
Ensured existence of index idx_project1_patch_polygon on project1_patch.
Ensured existence of table project1_pred_patch_latest for project 1.
Ensured existence of index idx_pred_patch_p1_latest_grid on project1_pred_patch_latest.
Ensured existence of table project1_pred_patch_last for project 1.
Ensured existence of index idx_pred_patch_p1_last_grid on project1_pred_patch_last.
Ensured existence of table project1_confusion_matrix_l8 for project 1.
Ensured existence of index idx_cm_p1_l8_nonpositive on project1_confusion_matrix_l8.
Ensured existence of table project1_confusion_matrix_l9 for project 1.
Ensured existence of index idx_cm_p1_l9_nonpositive on project1_confusion_matrix_l9.
Ensured existence of table project1_confusion_matrix_l10 for project 1.
Ensured existence of index idx_cm_p1_l10_nonpositive on project1_confusion_matrix_l10.
Ensured existence of table project1

In [7]:
# Add 10 label classes to the project.
# label_class_id=1 is the reserved "unassigned" class seeded at schema time.
# User-defined classes start at id=2.
with client.get_session() as session:
    lc_store = LabelClassStore(session)
    label_ids: dict[str, int] = {}
    for name, color in LABEL_CLASSES:
        lc = lc_store.create(project_id, name, color)
        label_ids[name] = lc["label_class_id"]
        print(f"  label_class_id={lc['label_class_id']:3d}  {name}  ({color})")

print(f"\n{len(label_ids)} label classes created.")

  label_class_id=  2  Epithelial  (#E74C3C)
  label_class_id=  3  Lymphocyte  (#3498DB)
  label_class_id=  4  Plasma Cell  (#9B59B6)
  label_class_id=  5  Macrophage  (#E67E22)
  label_class_id=  6  Neutrophil  (#F1C40F)
  label_class_id=  7  Eosinophil  (#1ABC9C)
  label_class_id=  8  Fibroblast  (#2ECC71)
  label_class_id=  9  Endothelial  (#E91E63)
  label_class_id= 10  Mast Cell  (#884400)
  label_class_id= 11  Tumor Cell  (#C0392B)

10 label classes created.


In [8]:
# Read image properties from the WSI via large_image.
import large_image

_ts = large_image.open(IMAGE_FILEPATH)
_meta = _ts.getMetadata()

BASE_MAG    = _meta["magnification"]
BASE_WIDTH  = _meta["sizeX"]
BASE_HEIGHT = _meta["sizeY"]

print(f"BASE_MAG={BASE_MAG}x  BASE_WIDTH={BASE_WIDTH}px  BASE_HEIGHT={BASE_HEIGHT}px")

BASE_MAG=40.0x  BASE_WIDTH=150784px  BASE_HEIGHT=71936px


In [9]:
# Register the whole-slide image.
with client.get_session() as session:
    img = ImageStore(session).create(
        project_id=project_id,
        name="placeholder_image",
        image_path=IMAGE_FILEPATH,
        base_mag=BASE_MAG,
        base_width=BASE_WIDTH,
        base_height=BASE_HEIGHT,
        deepzoom_tilesize=DEEPZOOM_TILESIZE,
    )

image_id = img["image_id"]
print(f"Image registered  image_id={image_id}  path={img['image_path']}")

Image registered  image_id=1  path=/opt/PatchSorter/mounts/fan_annotations/13_266069_040_003_L02_PAS.ndpi


In [10]:
# Extract patches from the GeoJSON file and load them into the database.
#
# All patches are assigned to the first user-defined label class by default.
# Change PATCH_LABEL_CLASS to the name of whichever class is appropriate,
# or supply an integer label_class_id directly.
PATCH_LABEL_CLASS = LABEL_CLASSES[0][0]  # "Tumor" — change as needed
patch_label_class_id = label_ids[PATCH_LABEL_CLASS]
for i in range(1):
    with client.get_session() as session:
        n_inserted = _makepatch_geojson(
            image_filepath=IMAGE_FILEPATH,
            geojson_filepath=GEOJSON_FILEPATH,
            project_id=project_id,
            image_id=image_id,
            label_class_id=patch_label_class_id,
            session=session,
            patch_size=PATCH_SIZE,
            downsample_factor=DOWNSAMPLE_FACTOR,
        )

print(f"Patches inserted: {n_inserted}  (label_class='{PATCH_LABEL_CLASS}', id={patch_label_class_id})")

Patches inserted: 61955  (label_class='Epithelial', id=2)


In [ ]:
db_mgr.clear_predictions(1)

In [11]:
# ---------------------------------------------------------------------------
# Initialize Ray cluster and start the deep learning actor
# ---------------------------------------------------------------------------

import ray
from patchsorter.dl.training import startup_dl_actor

In [12]:
# Start Ray cluster (connects to existing local cluster if already running)
if not ray.is_initialized():
    ray.init(dashboard_host="0.0.0.0")

print(f"Ray cluster address: {ray.get_runtime_context()}")

2026-06-17 13:23:03,402	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://172.21.0.3:8265 


Ray cluster address: <ray.runtime_context.RuntimeContext object at 0x70f045b3bb90>


/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [ ]:
# Create (or reuse) the named dl_actor and start the distributed training loop.
# This reads dl_num_workers and dl_patches_per_batch from the project settings.
project_id = 1
actor = startup_dl_actor(project_id)
print(f"DL actor started for project {project_id}")

DL actor started for project 1


(TrainController pid=300158) Requesting resources: {'GPU': 1} * 1
(TrainController pid=300158) Attempting to start training worker group of size 1 with the following resources: [{'GPU': 1}] * 1
(RayTrainWorker pid=300454) Setting up process group for: env:// [rank=0, world_size=1]
(TrainController pid=300158) Started training worker group of size 1: 
(TrainController pid=300158) - (ip=172.21.0.3, pid=300454) world_rank=0, local_rank=0, node_rank=0
(RayTrainWorker pid=300454) Loading pretrained weights from Hugging Face hub (timm/mobilenetv3_small_050.lamb_in1k)
(RayTrainWorker pid=300454) HTTP Request: HEAD https://huggingface.co/timm/mobilenetv3_small_050.lamb_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
(RayTrainWorker pid=300454) [timm/mobilenetv3_small_050.lamb_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
(RayTrainWorker pid=300454) Moving model to device: cuda:0
(RayTrainWorker pid=300454) Moving 

In [ ]:
ray.shutdown()

In [ ]:
with client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT p.shardid 
        FROM pg_dist_placement p
        JOIN pg_dist_node n ON p.groupid = n.groupid
        WHERE n.nodename = 'localhost'
        ORDER BY p.shardid;
    """)
    for row in cur.fetchall():
        print(row)

In [ ]:
with client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT * 
        FROM citus_shards 
        WHERE nodename = 'localhost';
    """)
    for row in cur.fetchall():
        print(row)

In [ ]:
with client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT shardid
        FROM citus_shards 
        WHERE nodename = 'localhost'
          AND table_name::text LIKE 'project1_patch%';
    """)
    shard_ids = [row[0] for row in cur.fetchall()]
    print(shard_ids)

In [ ]:
with client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT shardid
        FROM citus_shards 
        WHERE nodename = 'localhost'
          AND table_name::text LIKE 'project1_pred_patch_last%';
    """)
    shard_ids = [row[0] for row in cur.fetchall()]
    shard_ids.sort()
    print(shard_ids)

In [ ]:
with client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT shardid
        FROM citus_shards 
        WHERE nodename = 'localhost'
          AND table_name::text LIKE 'project1_pred_patch_last%';
    """)
    shard_ids = [row[0] for row in cur.fetchall()]
    shard_ids.sort()
    print(shard_ids)

In [ ]:
with client.get_connection() as conn:
    cur = conn.cursor()

    # Pick one shard id to inspect
    cur.execute("""
        SELECT shardid
        FROM citus_shards 
        WHERE nodename = 'localhost'
          AND table_name::text LIKE 'project1_pred_patch_last%'
        ORDER BY shardid
        LIMIT 1;
    """)
    shard_id = cur.fetchone()[0]
    shard_table = f"project1_pred_patch_last_{shard_id}"
    print(f"Inspecting shard: {shard_table}")

    cur.execute("""
        SELECT
            tgname        AS trigger_name,
            proname       AS function_name,
            tgenabled     AS enabled,
            tgisinternal  AS is_internal,
            pg_get_triggerdef(t.oid, true) AS full_definition
        FROM pg_trigger t
        JOIN pg_class   c ON c.oid = t.tgrelid
        JOIN pg_proc    p ON p.oid = t.tgfoid
        WHERE c.relname = %s
        ORDER BY tgname;
    """, (shard_table,))

    rows = cur.fetchall()
    if not rows:
        print("No triggers found on this shard.")
    for r in rows:
        print(r)